In [18]:
import torch
import math
import torch.nn as nn


In [14]:
def attention(Q,K,V,mask=None):
    """
    手搓 Attention（Scaled Dot-Product）

    参数:
        Q:Query,shape(batch,seq_len,d_k)
        K:Key,  shape(batch,seq_len,d_k)
        V:Value,shape(batch,seq_len,d_k)
        mask:可选,shape(batch,1,seq_len)或(batch,seq_len,seq_len)

    返回:
        output:加权后的输出,shape(batch,seq_len,d_v)
        attn_weights:注意力权重,shape(batch,seq_len,seq_len)
    """
    d_k=Q.size(-1)   #获取纬度，用于缩放

    #step1:算相似度scores= Q @ K ^ T/sqrt(d_k)
    scores=torch.matmul(Q,K.transpose(-2,-1))/math.sqrt(d_k)

    #step2:如果有mask,把mask=0的位置设为-inf（softmax后变成0）
    if mask is not None:
        scores=scores.masked_fill(mask==0,-1e9)

    #step3:softmax变成概率（每行和为1）
    attn_weights=torch.softmax(scores,dim=-1)

    #step4:用权重对V加权求和
    output=torch.matmul(attn_weights,V)
    #output shape:(batch,seq_len,d_v)

    return output , attn_weights
        

In [15]:
# 构造假数据
batch = 2      # 2 个句子
seq_len = 4    # 每个句子 4 个词
d_k = 8        # Q/K 的维度
d_v = 8        # V 的维度

Q = torch.randn(batch, seq_len, d_k)
K = torch.randn(batch, seq_len, d_k)
V = torch.randn(batch, seq_len, d_v)

# 跑 Attention
output, attn_weights = attention(Q, K, V)

print("Q/K/V shape:     ", Q.shape)
print("输出 shape:      ", output.shape)        # 应该是 (2, 4, 8)
print("注意力权重 shape:", attn_weights.shape)  # 应该是 (2, 4, 4)

# 验证：权重每行和为 1
print("\n权重每行和:")
print(attn_weights.sum(dim=-1))  # 应该全是 1.0

# 验证：scores 的 shape
scores_test = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
print("\nscores shape:    ", scores_test.shape)  # 应该是 (2, 4, 4)

Q/K/V shape:      torch.Size([2, 4, 8])
输出 shape:       torch.Size([2, 4, 8])
注意力权重 shape: torch.Size([2, 4, 4])

权重每行和:
tensor([[1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000]])

scores shape:     torch.Size([2, 4, 4])


In [8]:
# 模拟 padding mask：第 3、4 个位置是 padding（mask=0）
mask = torch.tensor([
    [1, 1, 1, 0],   # 第1个样本：前3个有效，第4个是padding
    [1, 1, 0, 0]    # 第2个样本：前2个有效，后
    2个是padding
]).unsqueeze(1)       # shape: (2, 1, 4)

output_masked, attn_masked = attention(Q, K, V, mask=mask)

print("带 mask 的注意力权重（第1个样本）:")
print(attn_masked[0])
# 你应该看到：第4列（最后一列）的值接近 0，因为 mask=0 的位置被忽略了

带 mask 的注意力权重（第1个样本）:
tensor([[0.6961, 0.1475, 0.1564, 0.0000],
        [0.0544, 0.8161, 0.1294, 0.0000],
        [0.4540, 0.0878, 0.4581, 0.0000],
        [0.6275, 0.2927, 0.0799, 0.0000]])


In [19]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, nhead=8):
        super().__init__()
        assert d_model % nhead == 0, "d_model 必须能被 nhead 整除"
        
        self.d_model = d_model
        self.nhead = nhead
        self.d_k = d_model // nhead   # 每个头的维度：512/8=64
        
        # 线性投影：把输入映射到 Q/K/V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        # 输出投影
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, x, mask=None):
        batch, seq_len, _ = x.shape
        
        # 1. 线性投影
        Q = self.W_q(x)  # (batch, seq, d_model)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # 2. 拆成 nhead 个头：reshape + transpose
        # (batch, seq, d_model) → (batch, nhead, seq, d_k)
        Q = Q.view(batch, seq_len, self.nhead, self.d_k).transpose(1, 2)
        K = K.view(batch, seq_len, self.nhead, self.d_k).transpose(1, 2)
        V = V.view(batch, seq_len, self.nhead, self.d_k).transpose(1, 2)
        
        # 3. 对每个头跑 Attention
        attn_out, attn_weights = attention(Q, K, V, mask)
        # attn_out shape: (batch, nhead, seq, d_k)
        
        # 4. 拼接多头：transpose + reshape
        attn_out = attn_out.transpose(1, 2).contiguous()
        attn_out = attn_out.view(batch, seq_len, self.d_model)
        
        # 5. 输出投影
        return self.W_o(attn_out)

# 测试
mha = MultiHeadAttention(d_model=512, nhead=8)
x = torch.randn(2, 10, 512)  # (batch=2, seq=10, dim=512)
out = mha(x)
print("Multi-Head 输出 shape:", out.shape)  # (2, 10, 512)

Multi-Head 输出 shape: torch.Size([2, 10, 512])


In [17]:
import torch
import torch.nn as nn
import math

# ========== 1. 手搓 Attention ==========
def attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    attn_weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    return output, attn_weights

# ========== 2. 测试 ==========
batch, seq_len, d_k = 2, 4, 8
Q = torch.randn(batch, seq_len, d_k)
K = torch.randn(batch, seq_len, d_k)
V = torch.randn(batch, seq_len, d_k)

output, attn_weights = attention(Q, K, V)

print("输出 shape:", output.shape)
print("权重 shape:", attn_weights.shape)
print("权重每行和:", attn_weights.sum(dim=-1))

# ========== 3. Multi-Head Attention（可选，看懂就行）==========
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, nhead=8):
        super().__init__()
        assert d_model % nhead == 0
        self.nhead = nhead
        self.d_k = d_model // nhead
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, x, mask=None):
        batch, seq_len, d_model = x.shape
        
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # 拆成多头
        Q = Q.view(batch, seq_len, self.nhead, self.d_k).transpose(1, 2)
        K = K.view(batch, seq_len, self.nhead, self.d_k).transpose(1, 2)
        V = V.view(batch, seq_len, self.nhead, self.d_k).transpose(1, 2)
        
        # 调用手搓的 attention
        attn_out, _ = attention(Q, K, V, mask)
        
        # 拼接回来
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch, seq_len, d_model)
        return self.W_o(attn_out)

# 测试 Multi-Head
mha = MultiHeadAttention(d_model=512, nhead=8)
x = torch.randn(2, 10, 512)
out = mha(x)
print("\nMulti-Head 输出 shape:", out.shape)

输出 shape: torch.Size([2, 4, 8])
权重 shape: torch.Size([2, 4, 4])
权重每行和: tensor([[1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000]])

Multi-Head 输出 shape: torch.Size([2, 10, 512])
